# 01 — Data Overview & Cleaning (per table)

**Project:** Arusto — *What Courses Should We Build Next?*  
**Later deliverable:** Marketing Demand Intelligence Model → dashboard

This notebook stays deliberately simple:

1. Look at each raw table on its own
2. Clean each table on its own
3. Check whether they *can* be joined later (key / coverage) — **but do not combine yet**

| File | What it is | Approx size |
|---|---|---|
| `linkedin_job_postings.csv` | job metadata | ~396 MB |
| `job_skills.csv` | NER skills per posting | ~642 MB |
| `job_summary.csv` | full job description text | ~4.8 GB |

> **Colab tip:** postings + skills usually fit in free Colab RAM. The full summary file often does not — we clean it carefully (sample first, then optional full pass / chunked write).


## 0. Setup


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 90)

# --- paths ---------------------------------------------------------------
# Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/Arusto_LinkedIn')

DATA_DIR = Path(".")
OUT_DIR = DATA_DIR / "cleaned"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POSTINGS_PATH = DATA_DIR / "linkedin_job_postings.csv"
SKILLS_PATH   = DATA_DIR / "job_skills.csv"
SUMMARY_PATH  = DATA_DIR / "job_summary.csv"

for p in [POSTINGS_PATH, SKILLS_PATH, SUMMARY_PATH]:
    if p.exists():
        print(f"OK       {p}  ({p.stat().st_size/1e6:.0f} MB)")
    else:
        print(f"MISSING  {p}")


In [ ]:
def peek_missing(df):
    """Small missingness table — reuse for each dataset."""
    missing = df.isna().sum()
    out = pd.DataFrame({
        "missing": missing,
        "missing_pct": (missing / len(df) * 100).round(3),
        "nunique": df.nunique(dropna=False),
        "dtype": df.dtypes.astype(str),
    }).sort_values("missing", ascending=False)
    return out


def save_df(df, name):
    """Prefer parquet; fall back to csv.gz if pyarrow isn't around."""
    parquet_path = OUT_DIR / f"{name}.parquet"
    csv_path = OUT_DIR / f"{name}.csv.gz"
    try:
        df.to_parquet(parquet_path, index=False)
        print(f"saved {parquet_path.name}  ({len(df):,} rows, {parquet_path.stat().st_size/1e6:.1f} MB)")
    except Exception as e:
        print(f"parquet failed ({e}); writing csv.gz")
        df.to_csv(csv_path, index=False, compression="gzip")
        print(f"saved {csv_path.name}  ({len(df):,} rows, {csv_path.stat().st_size/1e6:.1f} MB)")


def tf_to_bool(series):
    return series.astype("string").str.lower().map({"t": True, "f": False})


---
# A. `linkedin_job_postings.csv`

Spine table: title, company, location, level, type, scrape metadata.


## A1. Overview


In [ ]:
postings = pd.read_csv(POSTINGS_PATH)

print("shape:", postings.shape)
print("columns:", list(postings.columns))
print("memory (MB):", round(postings.memory_usage(deep=True).sum() / 1e6, 1))
postings.head(3)


In [ ]:
peek_missing(postings)


In [ ]:
# is job_link a usable primary key?
print("rows:", len(postings))
print("unique job_link:", postings["job_link"].nunique(dropna=False))
print("duplicate job_link rows:", postings["job_link"].duplicated().sum())
print("null job_link:", postings["job_link"].isna().sum())


In [ ]:
for col in ["search_country", "job_level", "job_type", "got_summary", "got_ner", "is_being_worked"]:
    print(f"\n=== {col} ===")
    print(postings[col].value_counts(dropna=False).head(12))


In [ ]:
print("first_seen min:", postings["first_seen"].min())
print("first_seen max:", postings["first_seen"].max())
print()
print("last_processed_time (raw sample):")
print(postings["last_processed_time"].head(3).tolist())
print()
print("job_title sample:")
print(postings["job_title"].head(8).tolist())


**Postings overview notes (fill in after you run):**

- Expected ~1.35M rows; `job_link` should be unique → good future join key
- `search_*` fields come from the *search query* used to collect the posting (not always identical to true job location / title)
- `got_summary` / `got_ner` tell us whether the other two tables are likely to have a matching row
- `first_seen` for this dump is a short Jan 2024 window → cross-sectional snapshot, not a long trend


## A2. Cleaning


In [ ]:
postings_clean = postings.copy()
n0 = len(postings_clean)

str_cols = [
    "job_link", "job_title", "company", "job_location",
    "search_city", "search_country", "search_position",
    "job_level", "job_type",
]
for c in str_cols:
    postings_clean[c] = postings_clean[c].astype("string").str.strip()

postings_clean = postings_clean.replace({"": pd.NA})

postings_clean["got_summary"] = tf_to_bool(postings_clean["got_summary"])
postings_clean["got_ner"] = tf_to_bool(postings_clean["got_ner"])
postings_clean["is_being_worked"] = tf_to_bool(postings_clean["is_being_worked"])

postings_clean["first_seen"] = pd.to_datetime(postings_clean["first_seen"], errors="coerce")
postings_clean["last_processed_time"] = pd.to_datetime(
    postings_clean["last_processed_time"], errors="coerce", utc=True
)

postings_clean["job_level"] = postings_clean["job_level"].str.title()
postings_clean["job_type"] = postings_clean["job_type"].str.title()

# must have a key + a title to be useful later
before = len(postings_clean)
postings_clean = postings_clean.dropna(subset=["job_link", "job_title"])
print(f"dropped missing job_link/job_title: {before - len(postings_clean)}")

before = len(postings_clean)
postings_clean = postings_clean.drop_duplicates(subset=["job_link"], keep="first")
print(f"dropped duplicate job_link: {before - len(postings_clean)}")

print(f"\npostings: {n0:,} → {len(postings_clean):,}")
postings_clean.head(2)


In [ ]:
# quick clean QA
print(postings_clean[["got_summary", "got_ner", "is_being_worked"]].dtypes)
print()
print(postings_clean["job_level"].value_counts(dropna=False))
print()
print(postings_clean["job_type"].value_counts(dropna=False))
print()
print("first_seen range:", postings_clean["first_seen"].min(), "→", postings_clean["first_seen"].max())


In [ ]:
save_df(postings_clean, "postings_clean")


---
# B. `job_skills.csv`

One row per posting (when NER ran). `job_skills` is a comma-separated string.


## B1. Overview


In [ ]:
skills = pd.read_csv(SKILLS_PATH)

print("shape:", skills.shape)
print("columns:", list(skills.columns))
print("memory (MB):", round(skills.memory_usage(deep=True).sum() / 1e6, 1))
skills.head(3)


In [ ]:
peek_missing(skills)


In [ ]:
print("rows:", len(skills))
print("unique job_link:", skills["job_link"].nunique(dropna=False))
print("duplicate job_link rows:", skills["job_link"].duplicated().sum())
blank = skills["job_skills"].isna().sum() + (skills["job_skills"].astype("string").str.strip() == "").sum()
print("null/blank job_skills:", int(blank))


In [ ]:
# how dense is the skills string?
skill_len = skills["job_skills"].astype("string").fillna("").str.len()
approx_n = skills["job_skills"].astype("string").fillna("").str.count(",") + 1
approx_n = approx_n.where(skill_len > 0, 0)

print("job_skills char length:")
print(skill_len.describe().round(1))
print()
print("approx #skills per row (comma count + 1):")
print(approx_n.describe().round(2))
print()
print("example raw cell:")
print(skills["job_skills"].dropna().iloc[0][:300])


## B2. Cleaning

Keep **two cleaned outputs** for this table alone:

- `skills_clean` — still one row per job, with a parsed `skill_list`
- `skills_long` — exploded to one skill per row (still *only* this table; no join to postings)


In [ ]:
skills_clean = skills.copy()
n0 = len(skills_clean)

skills_clean["job_link"] = skills_clean["job_link"].astype("string").str.strip()
skills_clean["job_skills"] = skills_clean["job_skills"].astype("string").str.strip()
skills_clean = skills_clean.replace({"": pd.NA})

before = len(skills_clean)
skills_clean = skills_clean.dropna(subset=["job_link", "job_skills"])
print(f"dropped missing link/skills: {before - len(skills_clean)}")

before = len(skills_clean)
skills_clean = skills_clean.drop_duplicates(subset=["job_link"], keep="first")
print(f"dropped duplicate job_link: {before - len(skills_clean)}")


def split_skills(raw):
    if pd.isna(raw):
        return []
    parts = [p.strip() for p in str(raw).split(",")]
    parts = [p for p in parts if p]
    return parts


skills_clean["skill_list"] = skills_clean["job_skills"].map(split_skills)
skills_clean["n_skills"] = skills_clean["skill_list"].map(len)

print(f"\nskills: {n0:,} → {len(skills_clean):,}")
print(skills_clean["n_skills"].describe().round(2))
skills_clean.head(2)


In [ ]:
skills_long = (
    skills_clean[["job_link", "skill_list"]]
    .explode("skill_list")
    .rename(columns={"skill_list": "skill"})
    .dropna(subset=["skill"])
    .copy()
)

skills_long["skill_raw"] = skills_long["skill"].astype("string").str.strip()
skills_long["skill"] = (
    skills_long["skill_raw"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# drop pure-punctuation leftovers from NER
skills_long = skills_long[skills_long["skill"].str.len() >= 1]
skills_long = skills_long[~skills_long["skill"].str.fullmatch(r"[\W_]+")]

before = len(skills_long)
skills_long = skills_long.drop_duplicates(subset=["job_link", "skill"])
print(f"skill mentions: {before:,} → {len(skills_long):,} (within-job dedupe)")
print("unique skills:", skills_long["skill"].nunique())


In [ ]:
top_skills = (
    skills_long["skill"]
    .value_counts()
    .head(20)
    .reset_index(name="mentions")
)
top_skills.columns = ["skill", "mentions"]
top_skills


In [ ]:
# export: keep a pipe-joined list so parquet/csv stay simple
skills_clean_export = skills_clean.copy()
skills_clean_export["skill_list_pipe"] = skills_clean_export["skill_list"].map(lambda xs: " | ".join(xs))
skills_clean_export = skills_clean_export.drop(columns=["skill_list"])

save_df(skills_clean_export, "skills_clean")
save_df(skills_long, "skills_long")


---
# C. `job_summary.csv`

~4.8GB full JD text. Overview with a sample first; cleaning can be sample-only or a chunked full pass.


## C1. Overview (sample first — safe on Colab)


In [ ]:
SUMMARY_SAMPLE_N = 20_000

summary_sample = pd.read_csv(SUMMARY_PATH, nrows=SUMMARY_SAMPLE_N)

print("sample shape:", summary_sample.shape)
print("columns:", list(summary_sample.columns))
summary_sample.head(2)


In [ ]:
peek_missing(summary_sample)


In [ ]:
summary_sample["_len"] = summary_sample["job_summary"].astype("string").fillna("").str.len()

print("summary length (sample):")
print(summary_sample["_len"].describe().round(1))
print()
print("unique job_link in sample:", summary_sample["job_link"].nunique())
print("dup job_link in sample:", summary_sample["job_link"].duplicated().sum())
print()
print("--- first summary preview ---")
print(str(summary_sample["job_summary"].iloc[0])[:500])


## C2. Cleaning

Default = clean the **sample** and save it (enough to understand text quality).  
Set `CLEAN_FULL_SUMMARY = True` only if you have RAM / time for a chunked full clean.


In [ ]:
def clean_summary_frame(df):
    out = df.copy()
    out["job_link"] = out["job_link"].astype("string").str.strip()
    out["job_summary"] = out["job_summary"].astype("string").str.strip()
    out = out.replace({"": pd.NA})
    out = out.dropna(subset=["job_link", "job_summary"])
    out = out.drop_duplicates(subset=["job_link"], keep="first")
    out["summary_len"] = out["job_summary"].str.len()
    # light text tidy — keep content, just collapse crazy whitespace
    out["job_summary"] = out["job_summary"].str.replace(r"[ \t]+", " ", regex=True)
    out["job_summary"] = out["job_summary"].str.replace(r"\n{3,}", "\n\n", regex=True)
    return out


summary_clean_sample = clean_summary_frame(summary_sample.drop(columns=["_len"], errors="ignore"))
print("sample cleaned:", len(summary_sample), "→", len(summary_clean_sample))
print(summary_clean_sample["summary_len"].describe().round(1))

save_df(summary_clean_sample, "summary_clean_sample")


In [ ]:
# Optional full clean — chunked so Colab / laptop don't explode
CLEAN_FULL_SUMMARY = False  # flip to True when you're ready
SUMMARY_CHUNKSIZE = 50_000

if CLEAN_FULL_SUMMARY:
    out_path = OUT_DIR / "summary_clean.csv.gz"
    first_chunk = True
    total_in = 0
    total_out = 0

    for chunk in pd.read_csv(SUMMARY_PATH, chunksize=SUMMARY_CHUNKSIZE):
        total_in += len(chunk)
        cleaned = clean_summary_frame(chunk)
        total_out += len(cleaned)
        cleaned.to_csv(
            out_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
            compression="gzip",
        )
        first_chunk = False
        print(f"processed {total_in:,} raw → {total_out:,} clean rows...", flush=True)

    print(f"\ndone. wrote {out_path} ({total_out:,} rows)")
else:
    print("skipped full summary clean (CLEAN_FULL_SUMMARY=False)")


---
# D. Join-key exploration only (no combine)

We are **not** merging tables here. Just answer: do they share a key, and how much would overlap?


In [ ]:
# candidate key from schema: job_link appears in all three files
print("candidate join key: job_link")
print()

postings_keys = set(postings_clean["job_link"].dropna().astype(str))
skills_keys = set(skills_clean["job_link"].dropna().astype(str))

print(f"unique keys — postings_clean: {len(postings_keys):,}")
print(f"unique keys — skills_clean:   {len(skills_keys):,}")
print(f"intersection (postings ∩ skills): {len(postings_keys & skills_keys):,}")
print(f"postings only: {len(postings_keys - skills_keys):,}")
print(f"skills only:   {len(skills_keys - postings_keys):,}")
print()
print(
    "share of cleaned postings that have a skills row:",
    f"{len(postings_keys & skills_keys) / len(postings_keys):.1%}",
)


In [ ]:
# compare against the flags already on postings — do they line up?
flag_ner_true = set(
    postings_clean.loc[postings_clean["got_ner"] == True, "job_link"].astype(str)
)

print("got_ner == True:", len(flag_ner_true))
print("in skills_clean:", len(skills_keys))
print("got_ner True ∩ skills:", len(flag_ner_true & skills_keys))
print("got_ner True but missing skills row:", len(flag_ner_true - skills_keys))
print("skills row but got_ner False/NA:", len(skills_keys - flag_ner_true))


In [ ]:
# summary key coverage — use sample by default; optional full key scan
SCAN_ALL_SUMMARY_KEYS = False  # True = iterate whole 4.8GB just for job_link (slow but RAM-light)

summary_keys_sample = set(summary_clean_sample["job_link"].dropna().astype(str))
print(f"summary keys in cleaned sample ({len(summary_clean_sample):,} rows): {len(summary_keys_sample):,}")
print(
    "overlap with postings (sample only — not a full-population estimate):",
    len(summary_keys_sample & postings_keys),
)

if SCAN_ALL_SUMMARY_KEYS:
    summary_keys = set()
    n = 0
    for chunk in pd.read_csv(SUMMARY_PATH, usecols=["job_link"], chunksize=100_000):
        summary_keys.update(chunk["job_link"].dropna().astype(str).str.strip())
        n += len(chunk)
        if n % 500_000 == 0:
            print(f"... scanned {n:,} summary rows", flush=True)
    print(f"\nunique summary job_link: {len(summary_keys):,}")
    print(f"postings ∩ summary: {len(postings_keys & summary_keys):,}")
    print(f"skills ∩ summary:   {len(skills_keys & summary_keys):,}")
    print(
        "share of postings with summary:",
        f"{len(postings_keys & summary_keys) / len(postings_keys):.1%}",
    )
else:
    # cheap proxy using the flag on postings
    print()
    print("full summary key scan skipped.")
    print(
        "proxy from postings.got_summary == True:",
        int((postings_clean["got_summary"] == True).sum()),
        f"({(postings_clean['got_summary'] == True).mean():.1%} of cleaned postings)",
    )


### Join-key takeaway

| Question | Answer we expect |
|---|---|
| Shared key? | Yes — `job_link` in all three tables |
| Primary key on postings? | Yes (unique after clean) |
| Skills / summary cardinality vs postings? | At most one row per `job_link` (1:1 style) |
| Ready to combine? | Key looks good; **actual merge waits for the next notebook** |


---
# E. Per-table cleaning checklist


In [ ]:
print("CLEANING CHECKLIST")
print("=" * 50)
print(f"A. postings_clean      : {len(postings_clean):,} rows")
print(f"B. skills_clean        : {len(skills_clean):,} rows")
print(f"B. skills_long         : {len(skills_long):,} rows  | unique skills: {skills_long['skill'].nunique():,}")
print(f"C. summary_clean_sample: {len(summary_clean_sample):,} rows  (full clean optional)")
print()
print("Outputs written to:", OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob("*")):
    print(f"  - {p.name:30} {p.stat().st_size/1e6:7.1f} MB")
print()
print("Next notebook: combine on job_link → EDA → demand scores → dashboard.")
